# C. Protein language-model embeddings and a linear classifier, in PyTorch

An optional extension of lecture 11 (book chapter 12): *representation
learning at scale*. Run it on [Google Colab](https://colab.research.google.com),
where PyTorch and Hugging Face `transformers` are preinstalled. A GPU is not
needed.

**Goals**

- Load a small pretrained protein language model, ESM-2
  (Rives et al. 2021; Lin et al. 2023), and turn every protein into one
  vector: the per-residue representations of a layer, averaged over the
  sequence.
- Train the same linear classifier (`nn.Linear` with the cross-entropy loss)
  on two representations: the amino-acid composition, and the ESM-2
  embeddings.
- See which layers of the model carry the useful information.

**Expected finding:** on the task of telling nuclear from cytoplasmic human
proteins, the composition gives about 65 % accuracy, and the embeddings of the
ESM-2 transformer layers 70 to 73 %. The first layer (the token embeddings,
averaged) is no better than the composition: the gain comes from the
attention layers, which give every residue a representation that depends on
its context.

The data are 500 reviewed human proteins from UniProtKB/Swiss-Prot
(release 2026_03, CC BY 4.0), 250 of them located in the nucleus and not in
the cytoplasm, 250 the other way round, each with experimental evidence. The
script `examples/_fetch_proteins.py` of the GlassNN repository wrote the file
and documents the query. The next cell downloads the file from GitHub, unless
it is already here.

In [ ]:
import pathlib
import urllib.request

DATA_URL = (
    "https://raw.githubusercontent.com/bioinfo-recetox/glassnn/"
    "main/examples/protein_localization.fasta"
)
if not pathlib.Path("protein_localization.fasta").exists():
    urllib.request.urlretrieve(DATA_URL, "protein_localization.fasta")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

lines = pathlib.Path("protein_localization.fasta").read_text().splitlines()
headers, sequences = lines[0::2], lines[1::2]
labels = [header.split()[1] for header in headers]
y = torch.tensor([label == "nucleus" for label in labels]).long()  # 1 = nucleus
lengths = [len(s) for s in sequences]
print(f"{len(sequences)} proteins, {int(y.sum())} nuclear")
print(f"lengths {min(lengths)} to {max(lengths)}")
print(headers[0], sequences[0][:60], "...")

## The baseline: amino-acid composition

The simplest fixed-length representation of a protein is the fraction of
each of the 20 amino acids: the features of the spectrum kernel of
lecture 5 with $k = 1$. (Pairs of amino acids, $k = 2$, do not help on this
data set.)

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def composition(sequence):
    return [sequence.count(a) / len(sequence) for a in AMINO_ACIDS]


X_composition = torch.tensor([composition(s) for s in sequences])
X_composition.shape

The classifier is logistic regression written as a network: one
`nn.Linear` layer and `cross_entropy`, trained by Adam on the whole training
set at once. With 500 proteins, one train/test split gives a noisy accuracy,
so the accuracy is averaged over a 5-fold cross-validation. The features are
standardized with the mean and standard deviation of the training folds
only.

In [ ]:
FOLDS = 5
fold = torch.from_numpy(np.random.default_rng(0).permutation(len(y)) % FOLDS)


def cv_accuracy(X):
    """Mean and standard deviation of the test accuracy over the folds."""
    accuracies = []
    for k in range(FOLDS):
        train, test = fold != k, fold == k
        mean, std = X[train].mean(dim=0), X[train].std(dim=0) + 1e-6
        Z = (X - mean) / std
        torch.manual_seed(0)
        model = nn.Linear(X.shape[1], 2)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.1)
        for _ in range(300):
            optimizer.zero_grad()
            nn.functional.cross_entropy(model(Z[train]), y[train]).backward()
            optimizer.step()
        with torch.no_grad():
            predicted = model(Z[test]).argmax(dim=1)
        accuracies.append((predicted == y[test]).float().mean().item())
    return np.mean(accuracies), np.std(accuracies)


baseline, baseline_std = cv_accuracy(X_composition)
print(f"composition: CV accuracy {baseline:.3f} ± {baseline_std:.3f}")

## ESM-2 embeddings

ESM-2 is a transformer encoder trained to predict masked amino acids on
millions of protein sequences (UniRef50), without any labels. The smallest
checkpoint has 6 layers and 8 million parameters, and represents each residue
by a vector of length 320. The model name and its exact version (`revision`,
a commit of the model repository on Hugging Face) are fixed, so the notebook
keeps giving the same numbers.

`add_pooling_layer=False` leaves out a pooling head that the checkpoint does
not contain; the loader still reports the masked-language-model head of the
checkpoint as unused, which is expected: only the encoder is needed here.

In [ ]:
from transformers import AutoTokenizer, EsmModel

MODEL = "facebook/esm2_t6_8M_UR50D"
REVISION = "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
esm = EsmModel.from_pretrained(MODEL, revision=REVISION, add_pooling_layer=False)
esm.eval()
n_parameters = sum(p.numel() for p in esm.parameters())
print(f"{esm.config.num_hidden_layers} layers of width {esm.config.hidden_size}")
print(f"{n_parameters} parameters")

The tokenizer adds a start token `<cls>` and an end token `<eos>` around
the residues. With `output_hidden_states=True` the model returns the
representation after the token embedding (layer 0) and after each of the 6
transformer layers. For every layer, the representations of the residues
(without the two extra tokens) are averaged into one vector per protein.
Proteins are processed one at a time, so no padding is needed; this takes
about a minute on a CPU.

In [ ]:
embeddings = []  # one (7, 320) tensor per protein
with torch.no_grad():
    for sequence in sequences:
        tokens = tokenizer(sequence, return_tensors="pt")
        hidden = esm(**tokens, output_hidden_states=True).hidden_states
        embeddings.append(torch.stack([h[0, 1:-1].mean(dim=0) for h in hidden]))
embeddings = torch.stack(embeddings)  # (proteins, layers, width)
embeddings.shape

In [ ]:
layer_accuracy = []
for layer in range(embeddings.shape[1]):
    accuracy, std = cv_accuracy(embeddings[:, layer])
    layer_accuracy.append(accuracy)
    print(f"ESM-2 layer {layer}: CV accuracy {accuracy:.3f} ± {std:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(layer_accuracy, marker="o", label="ESM-2 embedding, mean over residues")
ax.axhline(baseline, color="gray", linestyle="--", label="amino-acid composition")
ax.set_xlabel("layer (0 = token embedding)")
ax.set_ylabel("5-fold CV accuracy")
ax.legend()
plt.show()

The first two principal components of the last-layer embeddings already
separate the two classes partly, without any training on the labels.

In [ ]:
Z = embeddings[:, -1] - embeddings[:, -1].mean(dim=0)
U, S, Vh = torch.linalg.svd(Z, full_matrices=False)
pcs = Z @ Vh[:2].T
fig, ax = plt.subplots(figsize=(5, 4))
for value, name in [(1, "nucleus"), (0, "cytoplasm")]:
    ax.scatter(*pcs[y == value].T, s=8, alpha=0.6, label=name)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.legend()
plt.show()

**Finding.** The linear classifier is the same in both cases; only the
representation changes. The token embeddings, averaged (layer 0), are a
learned version of the composition and do no better. All transformer
layers do better, because each residue's representation now depends on the rest of
the sequence: the model has learned, from unlabeled sequences alone, features
that are useful for a task it was never trained on.

Two cautions. The folds are random, so a protein and a close homolog can end
up on the two sides of a split, which makes every accuracy here somewhat
optimistic; a careful benchmark splits by sequence similarity. And ESM-2 saw
these sequences (not their labels) during pre-training.

Compared with notebooks A and B: the model is put in evaluation mode with
`esm.eval()`, and the embeddings are computed inside `torch.no_grad()`,
because no gradient flows into the pretrained model; only `nn.Linear` is
trained. GlassNN has no counterpart of `transformers`: it does not load
pretrained models (book chapter 1).

## Rosetta table: GlassNN and PyTorch

Names, arguments and behaviour that differ between the two libraries (the same table as in chapter 12 of the GlassNN book). Everything not listed has the same name and meaning.

| Area | GlassNN | PyTorch | Difference |
|---|---|---|---|
| Tensors | `Tensor` | `torch.Tensor` | A float64 NumPy array is cast to the default dtype (PyTorch keeps float64). The graph is kept after `backward()`, so a second call works without `retain_graph=True`. There are no in-place operations (`+=`, item assignment); they raise an error. |
| Tensors | `Tensor.numpy` | `torch.Tensor.numpy()` | Returns a copy; PyTorch shares memory with the tensor. |
| Tensors | `backend.asarray` | `torch.as_tensor` | Floating-point input becomes the default dtype, so float64 data never mixes silently with float32 parameters. |
| Device and randomness | `backend.set_backend` | `tensor.to("cuda")`, `model.to("cuda")` | One global switch: all tensors created afterwards are on the GPU (CuPy). PyTorch places each tensor on a device. |
| Device and randomness | `backend.manual_seed` | `torch.manual_seed` | Returns a `numpy.random.Generator`, the same on CPU and GPU, so a seed gives identical runs on both; it is unseeded until called. PyTorch returns a `torch.Generator`, starts from a fixed seed, and draws different numbers on each device. |
| Device and randomness | `backend.from_dlpack`, `backend.to_dlpack` | `torch.from_dlpack`, `torch.utils.dlpack.to_dlpack` | Work on arrays, not tensors: wrap the result with `Tensor(...)`, pass `tensor.data`. |
| Data | `data.DataLoader` | `torch.utils.data.DataLoader` | Takes the arrays `X, y` directly instead of a `Dataset`; no workers, samplers or `collate_fn`. |
| Data | `data.one_hot` | `torch.nn.functional.one_hot` | Lives in `glassnn.data`. |
| Modules | `nn.Module.state_dict` | `Module.state_dict()` | Returns copies of the values, not tensors that share memory; parameters and buffers are listed module by module. |
| Modules | `nn.Module.register_buffer` | `Module.register_buffer(..., persistent=True)` | No `persistent` argument: every buffer is saved. |
| Layers | `nn.Linear` | `nn.Linear` | Extra argument `parametrization="ntk"` (NTK scaling, chapter 8); the default initialization is PyTorch's. |
| Layers | `nn.Conv1d`, `nn.Conv2d` | `nn.Conv1d`, `nn.Conv2d` | No `groups` and no `padding_mode` (zero padding only). |
| Layers | `functional.conv1d`, `functional.conv2d` | `F.conv1d`, `F.conv2d` | No `groups`; zero padding only. |
| Layers | `functional.max_pool1d`, `functional.max_pool2d`, `nn.MaxPool1d`, `nn.MaxPool2d` | `F.max_pool1d`, `nn.MaxPool1d`, ... | No `dilation`, `ceil_mode` or `return_indices`. |
| Layers | `functional.avg_pool1d`, `functional.avg_pool2d`, `nn.AvgPool1d`, `nn.AvgPool2d` | `F.avg_pool1d`, `nn.AvgPool1d`, ... | No `ceil_mode`; padding always counts in the mean (`count_include_pad=True`). |
| Layers | `nn.BatchNorm1d` | `nn.BatchNorm1d` | No `num_batches_tracked` buffer and no `momentum=None` (cumulative average). |
| Layers | `nn.Dropout` | `nn.Dropout` | No `inplace` argument. |
| Layers | `functional.dropout` | `F.dropout` | No `inplace` argument; an extra `generator=` argument. |
| Attention | `functional.scaled_dot_product_attention` | `F.scaled_dot_product_attention` | A boolean mask is `True` where attention is **forbidden** (PyTorch, in this function only: `True` = allowed). |
| Attention | `nn.MultiheadAttention` | `nn.MultiheadAttention` | Batch-first only, `(N, L, E)`. Three projections `q_proj`, `k_proj`, `v_proj` instead of one packed `in_proj_weight`, so `state_dict` names differ. No `kdim`, `vdim`, `add_bias_kv`, `add_zero_attn`. `is_causal=True` without a mask builds the causal mask. |
| Attention | `nn.TransformerEncoderLayer` | `nn.TransformerEncoderLayer` | Batch-first only. Defaults `norm_first=True` and `activation="gelu"` (PyTorch: `False`, `"relu"`); `activation` is a string. |
| Attention | `nn.TransformerEncoder` | `nn.TransformerEncoder` | No `enable_nested_tensor` or `mask_check`. |
| Attention | `nn.PositionalEncoding` | none (defined in PyTorch tutorials) | A GlassNN module: sinusoidal, batch-first, dropout 0 by default. |
| Losses | `functional.cross_entropy`, `nn.CrossEntropyLoss` | `F.cross_entropy`, `nn.CrossEntropyLoss` | Only `(N, C)` logits with integer targets: no class probabilities, `weight` or `ignore_index`. |
| Losses | `functional.mse_loss` | `F.mse_loss` | Different shapes raise an error (PyTorch broadcasts with a warning); `binary_cross_entropy_with_logits` does the same. |
| Optimizers | `optim.Optimizer` | `torch.optim.Optimizer` | One parameter group; the rate is `optimizer.lr` (PyTorch: `optimizer.param_groups[0]["lr"]`). |
| Optimizers | `optim.LinearWarmup` | `lr_scheduler.LinearLR(optimizer, start_factor, end_factor=1.0, total_iters=warmup_steps)` | The same schedule under a different name; all schedulers are also in `glassnn.optim`. |
| Optimizers | `optim.clip_grad_norm_` | `torch.nn.utils.clip_grad_norm_` | Lives in `glassnn.optim`; Euclidean norm only. |
| Tools | `gradcheck.gradcheck` | `torch.autograd.gradcheck` | Checks one vector-Jacobian product with fixed weights instead of the full Jacobian, and raises an error that names the element and both values. |
| Tools | `analysis.empirical_ntk`, `analysis.activation_stats`, `analysis.count_parameters` | none (`torch.func.jacrev`, forward hooks, `sum(p.numel() ...)`) | GlassNN helpers for chapters 4 and 8; the Colab notebooks show the PyTorch equivalents. |